# 1 Imports

In [16]:
import json
import numpy as np

from xgboost import XGBClassifier
import m2cgen as m2c

# 2 Funktionen definieren

In [17]:
def getScaleZP(modeldata, amountBits):
    min = 0.0
    max = 0.0
    
    for tree in modeldata['learner']['gradient_booster']['model']['trees']:
        nodescount = int(tree['tree_param']['num_nodes'])
        
        for i in range(nodescount):
            if int(tree['left_children'][i]) == -1:
                if float(tree['split_conditions'][i]) < min:
                    min = float(tree['split_conditions'][i])
                if float(tree['split_conditions'][i]) > max:
                    max = float(tree['split_conditions'][i])

    print(f"min: {min}, max: {max}")

    print(f'Amount of Bits: {amountBits} -> Skala: {(2**amountBits)-1} ')
    scale = ((2**amountBits)-1) / (max - min)
    zp = (-(round(scale*min))) - 128

    return scale, zp


def quantizeScores(modeldata, scale, zp):
    for tree in modeldata['learner']['gradient_booster']['model']['trees']:
        nodescount = int(tree['tree_param']['num_nodes'])
        
        for i in range(nodescount):
            if int(tree['left_children'][i]) == -1:
                score = float(tree['split_conditions'][i])
                tree['split_conditions'][i] = float(round(scale * score + zp))
                
    return modeldata

def portToC(model, filename):
    path = 'model-exports/plainC/'

    with open(path + filename,'w') as f:
        code = m2c.export_to_c(model)
        f.write(code)

    print(f'Model exported to: "{path + filename}"')

# 3 Exports Quantisieren

In [18]:
# Import model-jsons

with open('model-exports/aktuelle-exports/base_agmp.json') as f:
    agmp_json = json.load(f)

with open('model-exports/aktuelle-exports/base_harus.json') as f:
    harus_json = json.load(f)

with open('model-exports/aktuelle-exports/base_semu.json') as f:
    semu_json = json.load(f)

In [19]:
# Get scale and zero point for each model

agmp_scale, agmp_zp = getScaleZP(agmp_json, 8)
harus_scale, harus_zp = getScaleZP(harus_json, 8)
semu_scale, semu_zp = getScaleZP(semu_json, 8)

min: -0.26112887, max: 0.45713967
Amount of Bits: 8 -> Skala: 255 
min: -0.27594796, max: 0.8974817
Amount of Bits: 8 -> Skala: 255 
min: -0.82064354, max: 0.70868695
Amount of Bits: 8 -> Skala: 255 


In [20]:
# Quantize the models

agmp_json = quantizeScores(agmp_json, agmp_scale, agmp_zp)
harus_json = quantizeScores(harus_json, harus_scale, harus_zp)
semu_json = quantizeScores(semu_json, semu_scale, semu_zp)

In [23]:
# Export to JSON

with open('model-exports/aktuelle-exports/quantization/base_agmp_quantized.json', 'w') as f:
    json.dump(agmp_json, f, separators=(',', ':'))
with open('model-exports/aktuelle-exports/quantization/base_harus_quantized.json', 'w') as f:
    json.dump(harus_json, f, separators=(',', ':'))
with open('model-exports/aktuelle-exports/quantization/base_semu_quantized.json', 'w') as f:
    json.dump(semu_json, f, separators=(',', ':'))

# 4 Modelle porten

In [ ]:
agmp_clf = XGBClassifier()
agmp_clf.load_model('model-exports/aktuelle-exports/quantization/base_agmp_quantized.json')
agmp_clf.set_params(base_score=float(agmp_clf.base_score))
portToC(agmp_clf, 'base_agmp_quantized.c')

harus_clf = XGBClassifier()
harus_clf.load_model('model-exports/aktuelle-exports/quantization/base_harus_quantized.json')
harus_clf.set_params(base_score=float(harus_clf.base_score),
                     num_parallel_tree = 1)
portToC(harus_clf, 'base_harus_quantized.c')

semu_clf = XGBClassifier()
semu_clf.load_model('model-exports/aktuelle-exports/quantization/base_semu_quantized.json')
semu_clf.set_params(base_score=float(semu_clf.base_score))
portToC(semu_clf, 'base_semu_quantized.c')

Model exported to: "model-exports/plainC/base_agmp_quantized.c"
0.5
Model exported to: "model-exports/plainC/base_harus_quantized.c"
Model exported to: "model-exports/plainC/base_semu_quantized.c"


In [33]:
print(f"AGMP - scale:\t{agmp_scale} \tzp:{agmp_zp}")
print(f"HARUS - scale:\t{harus_scale} \tzp:{harus_zp}")
print(f"SEMU - scale:\t{semu_scale} \tzp:{semu_zp}")

AGMP - scale:	355.0204217492249 	zp:-35
HARUS - scale:	217.31170490440815 	zp:-68
SEMU - scale:	166.73962996709756 	zp:9
